# 基于LDA模型的手机评论情感分析
**数据来源**：京东商城 iPhone 15 用户评论（约10万条）  
**技术栈**：Python · requests · jieba · pandas · gensim · wordcloud  
**分析流程**：数据采集 → 清洗分词 → 情感分析 → LDA主题建模 → 可视化


## 1. 数据采集（爬虫）

In [ ]:
up = "./"

In [ ]:
# 导入库
import requests
from pymysql import *
import json
from lxml import etree
import os
import csv
import time
import pandas as pd

In [ ]:
# 创建爬虫类
file_list = [up + 'data/good{}.csv', up + 'data/bad{}.csv', up + 'data/middle{}.csv']
class spider(object):
    def __init__(self):
        self.page = 0
        # 2、1、0分别代表好、坏、中评
        self.score = [2, 1, 0]
        # url，注意京东的评论为动态加载，需要进入控制台界面寻找真正的url
        self.url = 'https://club.jd.com/comment/productPageComments.action?&productId=100019718263&score=1&sortType=5&page=0&pageSize=10&isShadowSku=0&rid=0&fold=1'
        self.headers = {
            'user-agent': 'Mozilla / 5.0(Windows NT 10.0;Win64;x64) AppleWebKit / 537.36(KHTML, likeGecko) Chrome / 107.0.0.0Safari / 537.36',
            "Cookie": "your_cookie_here"
        }

    def main(self,pid):
        for i, s in enumerate(self.score):
            # 数据初始化
            self.page = 0
            file_name = file_list[s].format(pid)
            self.create_file(file_name)
            result_rows = []
            # 由于平台限制，只能爬取100页评论
            while self.page < 100:
                print(f'正在爬取score值为{s}的第{self.page + 1}页')
                self.url = f'https://club.jd.com/comment/productPageComments.action?&productId={pid}&score={s}&sortType=5&page={self.page}&pageSize=10&isShadowSku=0&rid=0&fold=1'
                #print(self.url)
                time.sleep(2)
                self.page += 1
                response = requests.get(url=self.url, headers=self.headers, timeout=10)
                response = response.text
                #print(response)
                response_json = json.loads(response)
                comments = response_json['comments']
                for index, data in enumerate(comments):
                    try:
                        mId = data['id']
                    except:
                        mId = ''
                        print('数据错误')
                    try:
                        mName = data['nickname']
                    except:
                        mName = ''
                    try:
                        buyCount = data['extMap']['buyCount']
                    except:
                        buyCount = 0
                    try:
                        referenceName = data['referenceName']
                    except:
                        referenceName = ''
                    try:
                        productColor = data['productColor']
                    except:
                        productColor = ''
                    try:
                        productSize = data['productSize']
                    except:
                        productSize = ''
                    score = s
                    try:
                        content = data['content']
                    except:
                        content = ''
                    try:
                        creationTime = data['creationTime']
                    except:
                        creationTime = ''
                    result_data = [mId, mName, buyCount, referenceName, productColor,
                                   productSize, score, content, creationTime]
                    result_rows.append(result_data)
                self.save_to_csv(file_name, result_rows)

    def create_file(self, file_name):
        # 创建文件
        if not os.path.exists(file_name):
            with open(file_name, 'w', newline='', encoding='utf-8-sig') as f:
                writer = csv.writer(f)
                writer.writerow(['mId', 'mName', 'buyCount', 'referenceName',
                                 'productColor', 'productSize', 'score', 'content',
                                 'creationTime'])

    def save_to_csv(self, file_name, data):
        # 数据保存至本地csv文件
        with open(file_name, 'a',encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerows(data)

spider_obj = spider()

In [ ]:
spider_obj.main("100066896214")

## 2. 数据读取与合并

In [ ]:
proid = "100066896214 100066896370  100066896338 100068388535 100067259746 100066896526 100066896472 100068388451".split()
for pid in proid:
    spider_obj.main(pid)

In [ ]:
import glob
import re
import pandas as pd
import re
import numpy as np
import jieba.posseg as psg

In [ ]:
al_pd = []
for p in (glob.glob(up + "data/*")):
    df1 = pd.read_csv(p)
    df1["p"] = p
    al_pd.append(df1)

In [ ]:
df = pd.concat(al_pd)

## 3. 数据清洗与分词

In [ ]:
reviews=df[['content','score']].drop_duplicates()
content=reviews['content']

In [ ]:
strinfo=re.compile('[0-9a-zA-Z]|京东|苹果|外形外观|屏幕音效|拍照效果|运行速度|待机时间|手机|买|')
content=content.apply(lambda x:strinfo.sub('',x))

In [ ]:
worker=lambda s:[(x.word,x.flag) for x in psg.cut(s)]
seg_word=content.apply(worker)
seg_word

In [ ]:
#将词语转换为数据框形式，一列是词，一列是词语所在句子id，最后一列是词语在该句子中的位置
n_word=seg_word.apply(lambda x:len(x))#每条评论中词的个数
n_content=[[x+1]*y for x,y in zip(list(seg_word.index),list(n_word))]
index_content=sum(n_content,[])#将嵌套的列表展开，作为词所在评论的id
seg_word=sum(seg_word,[])
word=[x[0] for x in seg_word]#词
nature=[x[1] for x in seg_word]#词性
content_type=[[x]*y for x,y in zip(list(reviews['score']),list(n_word))]#评论类型
content_type=sum(content_type,[])
result=pd.DataFrame({'index_content':index_content,
                        'word':word,
                    'nature':nature,
                    'content_type':content_type})
#删除标点符号
result=result[result['nature']!='x']#x表示标点符号

In [ ]:
#删除停用词
stop_path=open(up + 'hit_stopwords.txt',encoding='utf-8')
stop=stop_path.readlines()
stop=[x.replace('\n','') for x in stop]
word=list(set(word)-set(stop))
result=result[result['word'].isin(word)]
#构造各词在对应评论的位置列
n_word=list(result.groupby(by=['index_content'])['index_content'].count())
index_word=[list(np.arange(0,y)) for y in n_word]
index_word=sum(index_word,[])#词语在该评论中的位置
result.loc[:,'index_word']=index_word

In [ ]:
result

## 4. 词频统计与词云

In [ ]:
ind=result[['n' in x for x in result['nature']]]['index_content'].unique()
result=result[[x in ind for x in result['index_content']]]
result = result[result["nature"]=='n']

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
frequencies=result.groupby(by=['word'])['word'].count()
frequencies=frequencies.sort_values(ascending=False)
background_image=plt.imread(up + '底图.png')
wordcloud=WordCloud(font_path='STHUPO.TTF',
                   max_words=100,
                   background_color='white',
                   mask=background_image)
my_wordcloud=wordcloud.fit_words(frequencies)
plt.imshow(my_wordcloud)
plt.axis('off')
plt.show()

In [ ]:
result.groupby(by=['word'])['word'].count()

## 5. 情感分析

In [ ]:
result.to_csv(up + 'word.csv',index=False,encoding='utf-8')

In [ ]:
# #读入正面，负面情感评价词
# pos_comment=pd.read_csv('good.txt',header=None,sep=r'\n',encoding='utf-8',engine='python')#正面评价词
# #负面评价词
# neg_comment=pd.read_csv('bad.txt',header=None,sep=r'\n',encoding='utf-8',engine='python')
#正面情感词语
pos_emotion=pd.read_csv(up + 'tsinghua.positive.gb.txt',header=None,sep=r'\n',encoding='gbk',engine='python')
#负面情感词
neg_emotion=pd.read_csv(up + 'tsinghua.negative.gb.txt',header=None,sep=r'\n',encoding='gbk',engine='python')
#合并情感词与评价词
positive=set(pos_emotion.iloc[:,0])
negative=set(neg_emotion.iloc[:,0])
intersection=positive&negative
intersection#正负面情感词表中相同的词语
positive=list(positive-intersection)
negative=list(negative-intersection)
positive=pd.DataFrame({'word':positive,
                      'weight':[1]*len(positive)})
negative=pd.DataFrame({'word':negative,
                      'weight':[-1]*len(negative)})
posneg=pd.concat([positive,negative],ignore_index=True)
posneg

In [ ]:
word=pd.read_csv(up +'word.csv')
data_posneg=posneg.merge(word,left_on='word',right_on='word',how='right')
data_posneg=data_posneg.sort_values(by=['index_content','index_word'])
data_posneg

In [ ]:
not_word=['不','没','无','非','莫','弗','毋','未','否','别','無','休','不是','不能','不可','没有','不用','不要','从没','不太']
df=pd.DataFrame(not_word)
df.to_csv(up + 'not_word.csv',encoding='utf-8')

In [ ]:
notdict=pd.read_csv(up + 'not_word.csv',names=['term'])
#处理否定修饰词
#构造新列，作为经过否定词修正后的情感值
data_posneg['amend_weight']=data_posneg['weight']
data_posneg['id']=np.arange(0,len(data_posneg))
only_inclination=data_posneg.dropna()#只保留有情感值的词语
only_inclination.index=np.arange(0,len(only_inclination))
index=only_inclination['id']
only_inclination

In [ ]:
only_inclination[only_inclination["amend_weight"]<0]

In [ ]:
only_inclination

In [ ]:
[[affective-1,affective-2]]

In [ ]:
# review['word'][[affective-1,affective-2]]
# affective=only_inclination['index_word'][i]
review['word']
review['word'][[1,2]]
review

In [ ]:
only_inclination[only_inclination["amend_weight"]<0]

In [ ]:
only_inclination[only_inclination['amend_weight']>0]

## 6. LDA 主题建模

In [ ]:
freq_neg=only_inclination[only_inclination['amend_weight']>0].groupby(by=['word'])['word'].count()

In [ ]:
result=result[['index_content','content_type','a_type']].drop_duplicates()
confusion_matrix=pd.crosstab(result['content_type'],result['a_type'],margins=True)#制作交差表
(confusion_matrix.iat[0,0]+confusion_matrix.iat[1,1])/confusion_matrix.iat[2,2]
#错误率
e=(confusion_matrix.iat[0,1]+confusion_matrix.iat[1,0])/confusion_matrix.iat[2,2]
#提取正面信息评论和负面信息评论
ind_pos=list(emotional_value[emotional_value['a_type']=='pos']['index_content'])
ind_neg=list(emotional_value[emotional_value['a_type']=='neg']['index_content'])
posdata=word[[i in ind_pos for i in word['index_content']]]
negdata=word[[i in ind_neg for i in word['index_content']]]

In [ ]:
emotional_value[emotional_value['a_type']=='pos']
emotional_value[emotional_value['a_type']=='neg']
freq_pos=posdata.groupby(by=['word'])['word'].count()
list(emotional_value[emotional_value['a_type']=='neg'])

In [ ]:
#绘制词云
#正面情感词词云
freq_pos=only_inclination[only_inclination['amend_weight']>0].groupby(by=['word'])['word'].count()
background_image=plt.imread(up + '底图.png')
wordcloud=WordCloud(font_path='STHUPO.TTF',
                   max_words=100,
                   background_color='white',
                   mask=background_image)
pos_wordcloud=wordcloud.fit_words(freq_pos)
plt.imshow(pos_wordcloud)
plt.axis('off')
plt.show()

In [ ]:
#负面情感词云
freq_neg=only_inclination[only_inclination['amend_weight']<0].groupby(by=['word'])['word'].count()
freq_neg=freq_neg.sort_values(ascending=False)
background_image=plt.imread(up + '底图.png')
wordcloud=WordCloud(font_path='STHUPO.TTF',
                   max_words=100,
                   background_color='white',
                   mask=background_image)
neg_wordcloud=wordcloud.fit_words(freq_neg)
plt.imshow(neg_wordcloud)
plt.axis('off')
plt.show()

In [ ]:
posdata.to_csv(up + 'posdata.csv',index=False,encoding='utf-8')
negdata.to_csv(up +'negdata.csv',index=False,encoding='utf-8')

In [ ]:
negdata

In [ ]:
import pandas as pd
import numpy as np
import re
import itertools
import matplotlib.pyplot as plt



#载入情感分析后的数据
pos_data=pd.read_csv(up + 'posdata.csv',encoding='utf-8')
neg_data=pd.read_csv(up + 'negdata.csv',encoding='utf-8')
from gensim import corpora,models
#建立词典
pos_dict=corpora.Dictionary([i] for i in pos_data['word'])#正面
neg_dict=corpora.Dictionary([i] for i in pos_data['word'])#负面
#建立语料库
pos_corpus=[pos_dict.doc2bow(j) for j in [[i] for i in pos_data['word']]]#正面
neg_corpus=[neg_dict.doc2bow(j) for j in [[i] for i in neg_data['word']]]#负面

In [ ]:
#构造主题数寻优函数
def cos(vector1,vetor2):
    dot_product=0.0
    normA=0.0
    normB=0.0
    for a,b in zip(vector1,vetor2):
        dot_product+=a*b
        normA+=a**2
        normB+=b**2
    if normA==0.0 or normB==0.0:
        return None
    else:
        return (dot_product/((normA*normB)**0.5))
#主题数寻优
def lda_k(x_corpus,x_dict):
    #初始化平均余弦相似度
    mean_similarity=[]
    mean_similarity.append(1)
    #循环生成主题并计算主题间相似度
    for i in np.arange(2,11):
        lda=models.LdaModel(x_corpus,num_topics=i,id2word=x_dict)#lda训练模型
        
        term=lda.show_topics(num_words=50)
        #提取各主题词
        top_word=[]
        for k in np.arange(i):
            top_word.append([''.join(re.findall('''(.*)''',i)) for i in term[k][1].split('+')])#列出所有词
        #构造词频向量
        word=sum(top_word,[])#列出所有词
        
        unique_word=set(word)#去除重复词

        #构造主题词列表，行表示主题号，列表示各主题词
        mat=[]
        for j in np.arange(i):
            top_w=top_word[j]
            mat.append(tuple([top_w.count(k) for k in unique_word]))
        print(mat)
        p=list(itertools.permutations(list(np.arange(i)),2))
        l=len(p)
        top_similarity=[0]
        for w in np.arange(l):
            vector1=mat[p[w][0]]
            vector2=mat[p[w][1]]
            top_similarity.append(cos(vector1,vector2))
        #计算平均余弦相似度
        mean_similarity.append(sum(top_similarity)/l)
    return mean_similarity
#计算主题平均余弦相似度
pos_k=lda_k(pos_corpus,pos_dict)
neg_k=lda_k(neg_corpus,neg_dict)

In [ ]:
from matplotlib.font_manager import FontProperties
font=FontProperties(size=14)
#解决中文显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
fig=plt.figure(figsize=(10,8))
ax1=fig.add_subplot(211)
ax1.plot(pos_k)
ax1.set_xlabel('正面评论LDA主题数寻优',fontproperties=font)
ax2=fig.add_subplot(212)
ax2.plot(neg_k)
ax2.set_xlabel('负面评论LDA主题数寻优',fontproperties=font)

## 8. 可视化输出

In [ ]:
pos_lda=models.LdaModel(pos_corpus,num_topics=3,id2word=pos_dict)
neg_lda=models.LdaModel(neg_corpus,num_topics=3,id2word=neg_dict)
pos_lda.print_topics(num_words=10)

In [ ]:
neg_lda.print_topics(num_words=10)

In [ ]:
import pysenti
import matplotlib.pyplot as plt
import jieba.analyse
%matplotlib inline
plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
from pandas import Series, DataFrame
def getS(x):
    if x>1:
        return "积极"
    if x < -1:
        return "消极"
    return "中性"

In [ ]:
df['情感'] = df['content'].apply(lambda x : pysenti.classify(x)['score'])
df['情感_cn'] = df['情感'] .apply(lambda x :getS(x) )

In [ ]:
df

In [ ]:
df.groupby(by=['情感_cn']).count().loc[:,['content']].plot(kind='pie',subplots=True)

In [ ]:
s = Series(df[  (df['情感'] >-30)  & (df['情感']<30)   ]['情感'])
s.hist( rwidth = 0.1, bins = 40,histtype = 'stepfilled')
plt.xlabel('情感值')
plt.ylabel('评论数量')
plt.show()

In [ ]:
df